In [1]:
# ==================================================
# Import Libraries
# ==================================================

from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().parents[1]

sys.path.insert(
    0,
    str(PROJECT_ROOT)
)


import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


from src.mlflow.tracking import MLflowTracker
from src.mlflow.utils import classification_metrics

d:\Subject\CV2026\Market Risk Classification\market-risk-classification\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==================================================
# Load Feature Store
# ==================================================

FEATURE_PATH = (
    r"D:\Subject\CV2026\Market Risk Classification\market-risk-classification\data\feature_store\BTCUSDT\BTCUSDT_feature_store.parquet"
)


df = pd.read_parquet(
    FEATURE_PATH
)


df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,...,close_lag_3,volume_lag_1,volume_lag_2,volume_lag_3,RSI_14_lag_1,RSI_14_lag_2,RSI_14_lag_3,MACD_lag_1,MACD_lag_2,MACD_lag_3
0,2017-08-17 05:39:00,4291.37,4291.37,4291.37,4291.37,0.134767,2017-08-17 05:39:59.999,578.335061,1,0.0,...,4320.00,0.098390,0.373933,0.019386,54.934206,54.934206,54.934206,1.040049,1.116026,1.196767
1,2017-08-17 05:40:00,4291.37,4291.37,4291.37,4291.37,0.000000,2017-08-17 05:40:59.999,0.000000,0,0.0,...,4320.00,0.134767,0.098390,0.373933,16.876822,54.934206,54.934206,-1.315204,1.040049,1.116026
2,2017-08-17 05:41:00,4291.37,4291.37,4291.37,4291.37,0.000000,2017-08-17 05:41:59.999,0.000000,0,0.0,...,4320.00,0.000000,0.134767,0.098390,16.876822,16.876822,54.934206,-3.145500,-1.315204,1.040049
3,2017-08-17 05:42:00,4291.37,4291.37,4291.37,4291.37,0.000000,2017-08-17 05:42:59.999,0.000000,0,0.0,...,4291.37,0.000000,0.000000,0.134767,16.876822,16.876822,16.876822,-4.543646,-3.145500,-1.315204
4,2017-08-17 05:43:00,4291.37,4291.37,4291.37,4291.37,0.000000,2017-08-17 05:43:59.999,0.000000,0,0.0,...,4291.37,0.000000,0.000000,0.000000,16.876822,16.876822,16.876822,-5.587280,-4.543646,-3.145500


In [3]:
# ==================================================
# Data Information
# ==================================================

print(df.shape)

df.info()

(1999017, 77)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1999017 entries, 0 to 1999016
Data columns (total 77 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   open_time               datetime64[ns]
 1   open                    float64       
 2   high                    float64       
 3   low                     float64       
 4   close                   float64       
 5   volume                  float64       
 6   close_time              datetime64[ns]
 7   quote_asset_volume      float64       
 8   number_of_trades        int64         
 9   taker_buy_base_volume   float64       
 10  taker_buy_quote_volume  float64       
 11  ignore                  int64         
 12  SMA_5                   float64       
 13  SMA_10                  float64       
 14  SMA_20                  float64       
 15  SMA_50                  float64       
 16  SMA_100                 float64       
 17  EMA_5                   float64 

In [4]:
# ==================================================
# Create Target
# ==================================================

df["target"] = (
    df["close"].shift(-1)
    >
    df["close"]
).astype(int)


df = df.dropna()


df["target"].value_counts()

target
0    1027439
1     971578
Name: count, dtype: int64

In [5]:
# ==================================================
# Drop Unnecessary Columns
# ==================================================

DROP_COLUMNS = [
    "open_time",
    "close_time",
    "target"
]


FEATURES = [
    col
    for col in df.columns
    if col not in DROP_COLUMNS
]


len(FEATURES)

75

In [6]:
# ==================================================
# Feature / Target
# ==================================================

X = df[FEATURES]

y = df["target"]


print(X.shape)
print(y.shape)

(1999017, 75)
(1999017,)


In [7]:
# ==================================================
# Missing Values
# ==================================================

df.isnull().sum().sum()

np.int64(0)

In [8]:
# ==================================================
# Train Test Split
# ==================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    shuffle=False
)


print(X_train.shape)
print(X_test.shape)

(1599213, 75)
(399804, 75)


In [9]:
# ==================================================
# Feature Scaling
# ==================================================

scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(
    X_train
)


X_test_scaled = scaler.transform(
    X_test
)

In [10]:
# ==================================================
# Initialize MLflow Tracker
# ==================================================

tracker = MLflowTracker()

In [11]:
# ==================================================
# Train Logistic Regression + MLflow
# ==================================================

model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    n_jobs=-1
)


with tracker.start_run(
    run_name="Logistic Regression Feature Store"
):

    model.fit(
        X_train_scaled,
        y_train
    )


    y_pred = model.predict(
        X_test_scaled
    )


    y_prob = model.predict_proba(
        X_test_scaled
    )[:, 1]


    metrics = classification_metrics(
        y_test,
        y_pred,
        y_prob
    )


    tracker.log_params(
        {
            "model": "LogisticRegression",
            "features": len(FEATURES),
            "max_iter": 1000,
            "random_state": 42
        }
    )


    tracker.log_metrics(
        metrics
    )


    tracker.log_model(
        model
    )


    tracker.generate_summary()

d:\Subject\CV2026\Market Risk Classification\market-risk-classification\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


MLflow training summary generated
Saved at: D:\Subject\CV2026\Market Risk Classification\market-risk-classification\artifacts\training_summary.txt


In [12]:
# ==================================================
# Metrics
# ==================================================

metrics

{'accuracy': 0.497458754789847,
 'precision': 0.4969333380160724,
 'recall': 0.7821374191570727,
 'f1_score': 0.607738323011925,
 'roc_auc': 0.4941864967149896}

In [13]:
# ==================================================
# MLflow Runs
# ==================================================

import mlflow


experiment = mlflow.get_experiment_by_name(
    "Market Risk Classification"
)


runs = mlflow.search_runs(
    experiment_ids=[
        experiment.experiment_id
    ]
)


columns = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc"
]


existing_columns = [
    col
    for col in columns
    if col in runs.columns
]


runs[
    existing_columns
]

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc
0,5527a31df31a499582e883b422d24e42,Logistic Regression Feature Store,0.497459,0.496933,0.782137,0.607738,0.494186
1,b001b6536bce4e50b61f667b72d53118,Logistic Regression Feature Store,0.497459,0.496933,0.782137,0.607738,0.494186
2,8b1a952038314ba3a384a78d3b3eb558,LSTM Baseline,0.510199,0.490223,0.361669,0.416247,0.509252
3,825287cd75de4ff1b86a3e713c36aa5e,LSTM Baseline,0.504229,0.488908,0.590417,0.534889,0.507697
4,ff6af802a7f44681b44412c7f043a5ad,Logistic Regression Baseline,0.522569,0.511390,0.230848,0.318101,0.519733
